## Issues
1. Capacity availability on certain date
   1. #of trucks available
2. Look at the transmode
   1. (Truck - Container) 40Ft truck  
3. Convert (shipment_plans) Units into pallets
4. Container should be limited by weight, area utilization, Volume
   1. Add-on: Conditions are Configurable from the user
   2. Calculate golden ratio (Currently A -> B)
5. Pull-in method


# Code

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from pandas import DataFrame, merge, concat

from numpy import floor, ceil, cumsum, where
from collections import defaultdict
from logging import getLogger
import math, time


## Classes

from objects.Axle import Axle
from objects.Container import Container
from objects.ContainerSummary import ContainerSummary
from objects.ContainerLoadingRules import ContainerLoadingRules
from objects.Dimension import Dimension
from objects.Pallet import Pallet
from objects.Position import Position
import uuid

##
from utils.visualize import container_visualization
from utils.measure_conversion import *



In [3]:
logger = getLogger("load_planner")

# Assumptions
1. Units in the pallet are homogeneous
2. Units are calculated from item quantity to pallets
3. Dimensions are measured in inches (Later converted to Feet / other metrics)
4. **Routes have been pre-planned**
5. 

# Solver
1. Decide what items to load based on:
- Delivery date, priority, Item name, 
- Convert units into pallets
- Confirm the #of units shipped & update 
2. Grouping logic:
- Combine all items

### Optimizers to look into
1. 
## Feature enhancements
1. Stock pull-in from future (Early shipping)
2. Axle based handling-unit 📦(container) positioning
3. 

In [4]:
### Writing results to Database ###
# Write results to tables:
# 1. Handling_unit
# 2. Handling_unit_content
# 3. Handling_unit_position
# 4. Transport_equipment_assignment
# 5. route_planned


In [5]:
### Verify loaded 🚚 truck_equipment_assignment & handling_unit positions inside the container  

## Establish connection with Neon Database

In [107]:
### NeonDB Connection
import database.helper as db_helper
db_conn = db_helper.create_connection()

## Read Data From Database

In [108]:
item_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.item_master", connection=db_conn)
lane_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.lane_master", connection=db_conn)
load_equipment_metadata_df = db_helper.fetch_data(sql="select * from inventory_management.public.load_equipment_metadata", connection=db_conn)
location_df = db_helper.fetch_data(sql="select * from inventory_management.public.location", connection=db_conn)
shipment_plans_df = db_helper.fetch_data(sql="select * from inventory_management.public.shipment_plans", connection=db_conn)
sku_uom_df = db_helper.fetch_data(sql="select * from inventory_management.public.sku_unit_of_measure", connection=db_conn)
transport_asset_df = db_helper.fetch_data(sql="select * from inventory_management.public.transport_asset", connection=db_conn)


## Create necessary features, calculations




In [109]:
sku_uom_df = pd.concat(
    [
        sku_uom_df,
        sku_uom_df['pallet_dimensions'].apply(pd.Series)
    ],
    axis=1
)

sku_uom_column_mapper = {x:x for x in sku_uom_df.columns}
sku_uom_column_mapper['height_mm'] = 'pallet_height_mm'
sku_uom_column_mapper['width_mm'] = 'pallet_width_mm'
sku_uom_column_mapper['length_mm'] = 'pallet_length_mm'
sku_uom_df.rename(columns=sku_uom_column_mapper, inplace=True)

In [10]:


sku_uom_df['sku_id'] = sku_uom_df['sku_id'].astype(str)
shipment_plans_df['sku_id'] = shipment_plans_df['sku_id'].astype(str)

In [11]:
shipment_plans_df = pd.merge(
    left=shipment_plans_df, 
    right=sku_uom_df[['sku_id', 'unit_count_in_pallet', 'pallet_height_mm', 'pallet_width_mm', 'pallet_length_mm', ]], 
    left_on=['sku_id'], 
    right_on=['sku_id'], 
    how='left'
)


In [13]:


shipment_plans_df['planned_quantity_units_per_pallet'] = shipment_plans_df['planned_quantity'] / shipment_plans_df['unit_count_in_pallet']

shipment_plans_df['item_weight_kg'] = (shipment_plans_df['weight_kg']/shipment_plans_df['planned_quantity']).round(2)


## Phase 1:



In [14]:
def build_fixed_container(
	load_equipment_metadata_df
):
	"""
	Build fixed 40FT container.
	
	Returns
	-------
	Container
	"""

	equipment_row = (

		load_equipment_metadata_df

		.loc[
			load_equipment_metadata_df[
				"equipment_name"
			]

			.str.upper()

			.str.contains(
				"40FT",
				na=False
			)
		]

		.iloc[0]
	)

	container = Container(

		containerId=str(
			equipment_row["equipment_id"]
		),

		containerType=
			equipment_row["equipment_name"],

		depth=
			equipment_row["length_mm"],

		width=
			equipment_row["width_mm"],

		height=
			equipment_row["height_mm"],

		internal_depth=
			equipment_row["internal_length_mm"],

		internal_width=
			equipment_row["internal_width_mm"],

		internal_height=
			equipment_row["internal_height_mm"],

		maxPayloadWeight=
			equipment_row["max_payload_weight_kg"],

		tareWeight=
			equipment_row["tare_weight_kg"],

		maxVolume=(

			equipment_row["internal_length_mm"]
			*
			equipment_row["internal_width_mm"]
			*
			equipment_row["internal_height_mm"]

		) / 1_000_000_000,

		door_width=
			equipment_row["door_width_mm"],

		door_height=
			equipment_row["door_height_mm"],

		pallets=[]
	)

	return container


In [15]:

container = build_fixed_container(load_equipment_metadata_df=load_equipment_metadata_df)

In [16]:
# Step 1 - Build Daily Demand
def build_daily_demand(
    shipment_plans_df: pd.DataFrame,
    planning_date
):

    df = shipment_plans_df.copy()

    planning_date = pd.to_datetime(
        planning_date
    )

    df["estimated_delivery_date"] = pd.to_datetime(
        df["estimated_delivery_date"],
        errors="coerce"
    )

    df = df[
        df["estimated_delivery_date"]
        <= planning_date
    ]

    df = df[
        df["planned_quantity"] > 0
    ]

    df["remaining_quantity"] = (
        df["planned_quantity"]
        -
        df["shipped_quantity"].fillna(0)
    )

    df = df[
        df["remaining_quantity"] > 0
    ]

    return df.reset_index(
        drop=True
    )

In [17]:
# Step 2 - Convert Units To Pallets

def convert_units_to_pallets(
    daily_demand_df
):

    df = daily_demand_df.copy()

    df["required_pallets"] = (
        df["remaining_quantity"]
        /
        df["unit_count_in_pallet"]
    )

    return df

In [18]:
# Step 3 - Apply Partial Pallet Rules

def apply_partial_pallet_rules(
    pallet_df,
    round_up_threshold=0.6
):

    df = pallet_df.copy()

    df["full_pallets"] = np.floor(
        df["required_pallets"]
    )

    df["fractional_pallet"] = (

        df["required_pallets"]

        -

        df["full_pallets"]
    )

    df["rounded_pallets"] = np.where(

        df["fractional_pallet"]

        >= round_up_threshold,

        np.ceil(
            df["required_pallets"]
        ),

        np.floor(
            df["required_pallets"]
        )
    )

    return df

In [19]:
# Step 4 - Build Shipment Buckets
def build_shipment_buckets(
    shipment_df
):

    bucket_df = (

        shipment_df

        .groupby(
            [
                "estimated_delivery_date",
                "origin_location_id",
                "destination_location_id",
                "sku_id"
            ],
            as_index=False
        )

        .agg(
            {
                "rounded_pallets": "sum",
                "item_weight_kg": "min",
                "unit_count_in_pallet": 'min',
                "priority": "max"
            }
        )
    )

    return bucket_df

In [20]:
# Step 5 - Build Load Queue

def build_load_queue(
    bucket_df
):

    return (

        bucket_df

        .sort_values(
            [
                "estimated_delivery_date",
                "priority",
                "rounded_pallets"
            ],
            ascending=[
                True,
                False,
                False
            ]
        )

        .reset_index(drop=True)
    )

In [21]:
# Step 7 - Container Build Solver

def solve_container_build(
    load_queue_df,
    container: Container
):

    container_builds = []

    container_number = 1

    current_weight = 0

    current_pallets = []

    max_weight = (
        container.maxPayloadWeight
    )

    for _, row in load_queue_df.iterrows():

        pallet_weight = (
            (row['item_weight_kg'] * row['unit_count_in_pallet'])
            /
            max(
                row["rounded_pallets"],
                1
            )
        )

        pallet_count = int(
            row["rounded_pallets"]
        )
        print("len of pallet: " f"{pallet_count}", "max_weight: " f"{max_weight}", "pallet_weight: " f"{pallet_weight}")
        for _ in range(
            pallet_count
        ):

            if (

                current_weight
                + pallet_weight
                >
                max_weight

            ):
                container_builds.append({
                    "container_id":
                        f"CONT_{container_number}",
                    "pallets":
                        current_pallets
                })

                container_number += 1
                current_pallets = []
                current_weight = 0

            current_pallets.append({

                "sku_id":
                    row["sku_id"],

                "weight_kg":
                    pallet_weight,

                "destination_location_id":
                    row["destination_location_id"]
            })

            current_weight += (
                pallet_weight
            )

    if current_pallets:

        container_builds.append({

            "container_id":

                f"CONT_{container_number}",

            "pallets":

                current_pallets
        })

    return container_builds


In [22]:
# Step 8 - Build Physical Pallets

def build_pallets(
    container_builds,
    shipment_plans_df
):

    pallets = []

    for container in container_builds:

        for pallet in container["pallets"]:

            sku = pallet["sku_id"]

            shipment_row = (

                shipment_plans_df

                .loc[
                    shipment_plans_df[
                        "sku_id"
                    ]
                    == sku
                ]

                .iloc[0]
            )

            pallets.append({

                "container_id":
                    container[
                        "container_id"
                    ],

                "pallet_id":

                    str(
                        uuid.uuid4()
                    ),

                "sku_id":
                    sku,

                "weight_kg":
                    pallet[
                        "weight_kg"
                    ],

                "length_mm":

                    shipment_row[
                        "pallet_length_mm"
                    ],

                "width_mm":

                    shipment_row[
                        "pallet_width_mm"
                    ],

                "height_mm":

                    shipment_row[
                        "pallet_height_mm"
                    ],

                "position_x": 0,

                "position_y": 0,

                "position_z": 0
            })

    return pd.DataFrame(
        pallets
    )




In [23]:
# Step 9 - Place Pallets


def place_pallets(
    pallet_df,
    container: Container
):

    df = pallet_df.copy()

    current_x = 0
    current_z = 0

    row_depth = 0

    container_length = (
        container.internal_depth
        
    )

    container_width = (
        container.internal_width
        
    )

    for idx in df.index:

        pallet_length = (
            df.loc[
                idx,
                "length_mm"
            ]
        )

        pallet_width = (
            df.loc[
                idx,
                "width_mm"
            ]
        )

        if (

            current_z
            + pallet_width

            >

            container_width

        ):

            current_x += row_depth

            current_z = 0

            row_depth = 0

        if (

            current_x
            + pallet_length
            >
            container_length

        ):
            pass
            # raise Exception(
            #     "Container full push to next container. Fix: Weight / volume prior selection"
            # )

        df.loc[
            idx,
            "position_x"
        ] = current_x

        df.loc[
            idx,
            "position_z"
        ] = current_z

        current_z += pallet_width

        row_depth = max(
            row_depth,
            pallet_length
        )

    return df

In [24]:
# Step 10 - Center of Gravity

def calculate_center_of_gravity(
    pallet_df
):

    total_weight = (
        pallet_df[
            "weight_kg"
        ].sum()
    )

    cg_x = (

        pallet_df[
            "weight_kg"
        ]

        *

        pallet_df[
            "position_x"
        ]

    ).sum() / total_weight

    cg_z = (

        pallet_df[
            "weight_kg"
        ]

        *

        pallet_df[
            "position_z"
        ]

    ).sum() / total_weight

    return {

        "cg_x": cg_x,

        "cg_z": cg_z
    }

In [25]:
# Run Pipeline

planning_date = "2026-06-10"

daily_demand_df = build_daily_demand(
    shipment_plans_df,
    planning_date
)

pallet_req_df = convert_units_to_pallets(
    daily_demand_df
)

shipment_bucket_df = (
    apply_partial_pallet_rules(
        pallet_req_df
    )
)

bucket_df = build_shipment_buckets(
    shipment_bucket_df
)

load_queue_df = build_load_queue(
    bucket_df
)

container = build_fixed_container(
    load_equipment_metadata_df
)

container_builds = (
    solve_container_build(
        load_queue_df,
        container
    )
)


len of pallet: 132 max_weight: 25000.0 pallet_weight: 909.6733333333334
len of pallet: 78 max_weight: 25000.0 pallet_weight: 919.4676923076922
len of pallet: 72 max_weight: 25000.0 pallet_weight: 4492.442500000001
len of pallet: 30 max_weight: 25000.0 pallet_weight: 4286.46
len of pallet: 29 max_weight: 25000.0 pallet_weight: 49686.98482758621
len of pallet: 28 max_weight: 25000.0 pallet_weight: 44132.64
len of pallet: 28 max_weight: 25000.0 pallet_weight: 64664.61428571429
len of pallet: 22 max_weight: 25000.0 pallet_weight: 22306.96363636364
len of pallet: 21 max_weight: 25000.0 pallet_weight: 27636.48
len of pallet: 17 max_weight: 25000.0 pallet_weight: 106592.18823529412
len of pallet: 16 max_weight: 25000.0 pallet_weight: 51338.88
len of pallet: 16 max_weight: 25000.0 pallet_weight: 113254.2
len of pallet: 15 max_weight: 25000.0 pallet_weight: 2779.6
len of pallet: 14 max_weight: 25000.0 pallet_weight: 129433.37142857142
len of pallet: 13 max_weight: 25000.0 pallet_weight: 139277.

In [26]:
container_builds[:5]

[{'container_id': 'CONT_1',
  'pallets': [{'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
    'destination_location_id': '6062'},
   {'sku_id': '203277',
    'weight_kg': 909.6733333333334,
 

In [27]:
len(container_builds[:5])

5

In [28]:

pallet_df = build_pallets(
    container_builds[:5],
    shipment_plans_df
)

pallet_df = place_pallets(
    pallet_df,
    container
)

cg = calculate_center_of_gravity(
    pallet_df
)

print(cg)


{'cg_x': np.float64(40543.65375255374), 'cg_z': np.float64(504.19743142204663)}


In [29]:
from copy import deepcopy

def build_container_objects(
    container_builds,
    pallet_position_df,
    container_template
):
    """
    Build Pydantic Container objects.

    ```
    Returns
    -------
    List[Container]
    """

    containers = []

    for build in container_builds:

        container_id = build["container_id"]

        container_pallets = (

            pallet_position_df

            [
                pallet_position_df[
                    "container_id"
                ]

                ==

                container_id
            ]
        )

        container = deepcopy(
            container_template
        )

        container.containerId = (
            container_id
        )

        container.pallets = []

        total_weight = 0
        total_volume = 0

        for _, pallet_row in (

            container_pallets

            .iterrows()
        ):

            pallet = Pallet(

                label=str(
                    pallet_row[
                        "sku_id"
                    ]
                ),

                color="#4CAF50",

                dimensions=Dimension(

                    depth=int(
                        pallet_row[
                            "length_mm"
                        ]
                    ),

                    width=int(
                        pallet_row[
                            "width_mm"
                        ]
                    ),

                    height=int(
                        pallet_row[
                            "height_mm"
                        ]
                    )
                ),

                position=Position(

                    x=int(
                        pallet_row[
                            "position_x"
                        ]
                    ),

                    y=int(
                        pallet_row[
                            "position_y"
                        ]
                    ),

                    z=int(
                        pallet_row[
                            "position_z"
                        ]
                    )
                )
            )

            container.pallets.append(
                pallet
            )

            total_weight += (
                pallet_row[
                    "weight_kg"
                ]
            )

            # total_volume += (
            #     pallet_row[
            #         "volume_m3"
            #     ]
            # )

        container.summary.totalPallets = (
            len(
                container.pallets
            )
        )

        container.summary.totalWeight = (
            round(
                total_weight,
                2
            )
        )

        container.summary.totalVolume = (
            round(
                total_volume,
                2
            )
        )

        containers.append(
            container
        )

    return containers


In [30]:
containers = build_container_objects(
    container_builds=container_builds[:5],
    pallet_position_df=pallet_df,
    container_template=container
)

Container(containerId='2', containerType='CONTAINER 40FT GP', depth=12192.0, width=2438.0, height=2591.0, internal_depth=12031.0, internal_width=2352.0, internal_height=2393.0, maxPayloadWeight=25000.0, tareWeight=0.0, maxVolume=67.714510416, unit='mm', door_width=2340.0, door_height=2280.0, axles=[Axle(axleId='DEFAULT', maxWeight=30000, positionX=1371.6)], pallets=[], summary=ContainerSummary(shipmentId='', routeId='', origin='', destinationInSequence=[], totalPallets=0, totalWeight=0, totalVolume=0), loadingRules=ContainerLoadingRules(allowStacking=False, maxStackHeight=1, lifoEnabled=True, fragileSeparation=True, hazmatSegregation=True, centerGravityThreshold=15))

In [38]:
# for _container_ in containers:
# for _container_ in containers:
container_visualization(data=containers[0])
container_visualization(data=containers[1])

http://localhost:5173/container-visualization?data=%7B%22containerId%22%3A%20%22CONT_1%22%2C%20%22containerType%22%3A%20%22CONTAINER%2040FT%20GP%22%2C%20%22depth%22%3A%2012192.0%2C%20%22width%22%3A%202438.0%2C%20%22height%22%3A%202591.0%2C%20%22internal_depth%22%3A%2012031.0%2C%20%22internal_width%22%3A%202352.0%2C%20%22internal_height%22%3A%202393.0%2C%20%22maxPayloadWeight%22%3A%2025000.0%2C%20%22tareWeight%22%3A%200.0%2C%20%22maxVolume%22%3A%2067.714510416%2C%20%22unit%22%3A%20%22mm%22%2C%20%22door_width%22%3A%202340.0%2C%20%22door_height%22%3A%202280.0%2C%20%22axles%22%3A%20%5B%7B%22axleId%22%3A%20%22DEFAULT%22%2C%20%22maxWeight%22%3A%2030000%2C%20%22positionX%22%3A%201371.6%7D%5D%2C%20%22pallets%22%3A%20%5B%7B%22dimensions%22%3A%20%7B%22depth%22%3A%201219%2C%20%22width%22%3A%201016%2C%20%22height%22%3A%201277%7D%2C%20%22position%22%3A%20%7B%22x%22%3A%200%2C%20%22y%22%3A%200%2C%20%22z%22%3A%200%7D%2C%20%22label%22%3A%20%22203277%22%2C%20%22color%22%3A%20%22%234CAF50%22%7D%2C%20%7B%

http://localhost:5173/container-visualization?data=%7B%22containerId%22%3A%20%22CONT_2%22%2C%20%22containerType%22%3A%20%22CONTAINER%2040FT%20GP%22%2C%20%22depth%22%3A%2012192.0%2C%20%22width%22%3A%202438.0%2C%20%22height%22%3A%202591.0%2C%20%22internal_depth%22%3A%2012031.0%2C%20%22internal_width%22%3A%202352.0%2C%20%22internal_height%22%3A%202393.0%2C%20%22maxPayloadWeight%22%3A%2025000.0%2C%20%22tareWeight%22%3A%200.0%2C%20%22maxVolume%22%3A%2067.714510416%2C%20%22unit%22%3A%20%22mm%22%2C%20%22door_width%22%3A%202340.0%2C%20%22door_height%22%3A%202280.0%2C%20%22axles%22%3A%20%5B%7B%22axleId%22%3A%20%22DEFAULT%22%2C%20%22maxWeight%22%3A%2030000%2C%20%22positionX%22%3A%201371.6%7D%5D%2C%20%22pallets%22%3A%20%5B%7B%22dimensions%22%3A%20%7B%22depth%22%3A%201219%2C%20%22width%22%3A%201016%2C%20%22height%22%3A%201277%7D%2C%20%22position%22%3A%20%7B%22x%22%3A%2015847%2C%20%22y%22%3A%200%2C%20%22z%22%3A%201016%7D%2C%20%22label%22%3A%20%22203277%22%2C%20%22color%22%3A%20%22%234CAF50%22%7D%2C

In [82]:
containers[1].pallets

[Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=15847, y=0, z=1016), label='203277', color='#4CAF50'),
 Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=17066, y=0, z=0), label='203277', color='#4CAF50'),
 Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=17066, y=0, z=1016), label='203277', color='#4CAF50'),
 Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=18285, y=0, z=0), label='203277', color='#4CAF50'),
 Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=18285, y=0, z=1016), label='203277', color='#4CAF50'),
 Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=19504, y=0, z=0), label='203277', color='#4CAF50'),
 Pallet(dimensions=Dimension(depth=1219, width=1016, height=1277), position=Position(x=19504, y=0, z=1016), label='203277', color='#4CAF50'),
 Pallet(dimensi

## Create Links for the following
1. Transport equipment assignment 🚚
   1. (transport_asset_df + load_equipment) [🛻+📦 record] 
2. 